# ARIA — vLLM Inference Server (vast.ai)

Serves the merged BF16 safetensors as an OpenAI-compatible API on port 8000.
The ARIA Dash app connects via `OLLAMA_BASE_URL=http://<host>:8000/v1` — same interface as Ollama.

**Prerequisites (done in the training notebook before running this):**
- Cell 8b-pre: merge LoRA + save BF16 safetensors to `GGUF_DIR` (`/workspace/aria-v16-gguf`)
- Cell 8b-upload: upload all safetensors + tokenizer files to `HF_REPO`

**Recommended vast.ai instance:**
- Image: `pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime` (or any CUDA 11.8+ image)
- GPU: RTX 3090 24 GB (minimum for BF16 4B) — A100 preferred
- Disk: 30 GB free on `/workspace`

**Cell sequence:** 1 → 2 → 3 → *restart kernel* → 1 → 3b → 4 → 5 → 6 → 7 → 8

> **Cell 3b** redirects to the local training output if you are on the same instance where you trained.
> Cell 4 then detects the files and skips the HF download automatically.

## Cell 1 — Configuration
Edit `HF_TOKEN` and `HF_REPO` before running.

In [ ]:
import os, sys
from pathlib import Path

os.environ['HF_HOME'] = '/workspace/huggingface'
os.environ['HF_TOKEN'] = ''          # ← paste your HuggingFace token here (never commit)

HF_REPO       = 'speri420/aria-v16'              # ← HF repo with BF16 safetensors
TRAINING_DIR  = Path('/workspace/aria-v16-gguf') # ← where training notebook saves merged BF16
MODEL_DIR     = Path('/workspace/aria-vllm-model')
MODEL_NAME    = 'aria-v16'                       # served model name → set OLLAMA_MODEL=aria-v16 in app
PORT          = 8000
MAX_SEQ_LEN   = 16384  # prompt 2k+ tokens + MAX_TOKENS_TOOL 6144 — needs headroom

HF_TOKEN = os.environ.get('HF_TOKEN', '')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repo:         {HF_REPO}')
print(f'Training dir: {TRAINING_DIR}')
print(f'Model dir:    {MODEL_DIR}')
print(f'Serves:       {MODEL_NAME}  on port {PORT}')
print(f'Token:        {"set" if HF_TOKEN else "NOT SET — required if downloading from private HF repo"}')

## Cell 2 — GPU / disk check

In [ ]:
import subprocess, torch

print('=== GPU ===')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'{props.name}  {vram_gb:.1f} GB VRAM')
    if vram_gb < 10:
        print('WARNING: <10 GB VRAM — BF16 4B model needs ~10 GB minimum. Use GGUF+Ollama instead.')
    elif vram_gb < 16:
        print('CAUTION: may be tight. Add --enforce-eager to the vLLM launch command if OOM.')
    else:
        print('OK — enough VRAM for BF16 4B + KV cache.')
else:
    raise RuntimeError('No CUDA GPU detected — vLLM requires GPU.')

print('\n=== Disk ===')
subprocess.run('df -h /workspace', shell=True)
# Need: ~8 GB model download + HF cache + pip packages (~4 GB)

## Cell 3 — Install vLLM

Takes 3–10 min on a fresh instance. **Restart the kernel after this cell completes,
then re-run Cell 1 before continuing.**

> **Requires a vast.ai instance with Max CUDA 13.x** (NVIDIA driver 570+).
> After install, also run:
> ```bash
> pip install nvidia-cuda-runtime --quiet
> ```
> This installs `libcudart.so.13` to `/venv/main/lib/python3.12/site-packages/nvidia/cu13/lib/`
> which Cell 5 adds to `LD_LIBRARY_PATH` automatically.

In [ ]:
import subprocess, sys

for pkg, label in [
    (['vllm', 'huggingface_hub', 'openai', '--upgrade'], 'vLLM + deps'),
    (['nvidia-cuda-runtime'], 'CUDA 13 runtime (libcudart.so.13)'),
]:
    print(f'Installing {label} ...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkg,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'Install failed for {label}')
    print(r.stdout[-200:] or 'OK.')

print('\n*** Restart the kernel now, then re-run Cell 1 and continue from Cell 4. ***')

## Cell 3b — Use local training output (skip HF download if on same instance)

If you trained aria-v16 on **this same vast.ai instance**, the merged BF16 safetensors are
already at `TRAINING_DIR` (`/workspace/aria-v16-gguf`). This cell redirects `MODEL_DIR` there
so Cell 4 finds them immediately and skips the HF download.

Run this cell after the kernel restart and Cell 1 re-run, before Cell 4.

In [ ]:
sf_local = list(TRAINING_DIR.glob('*.safetensors'))
if sf_local:
    total_gb = sum(f.stat().st_size for f in sf_local) / 1e9
    MODEL_DIR = TRAINING_DIR
    print(f'Local training output found: {len(sf_local)} safetensors ({total_gb:.1f} GB)')
    print(f'MODEL_DIR → {MODEL_DIR}')
    print('Cell 4 will detect these files and skip the HF download.')
else:
    print(f'No safetensors at {TRAINING_DIR} — using MODEL_DIR={MODEL_DIR}')
    print('Cell 4 will download from HF.')

## Cell 4 — Download BF16 safetensors from HuggingFace

Downloads model weights, config, and tokenizer. Skips `.gguf` files.
Idempotent — skips download if safetensors already present.

In [ ]:
from huggingface_hub import snapshot_download

sf_files = list(MODEL_DIR.glob('*.safetensors'))
if sf_files:
    total_gb = sum(f.stat().st_size for f in sf_files) / 1e9
    print(f'Model already at {MODEL_DIR}  ({total_gb:.1f} GB safetensors) — skipping download.')
else:
    print(f'Downloading {HF_REPO} -> {MODEL_DIR} ...')
    print('(BF16 4B model ~8 GB — takes a few minutes on fast vas.ai uplink)')
    snapshot_download(
        repo_id=HF_REPO,
        local_dir=str(MODEL_DIR),
        token=HF_TOKEN or None,
        ignore_patterns=['*.gguf', '*.bin'],   # skip GGUF quants if present
    )
    sf_files = list(MODEL_DIR.glob('*.safetensors'))
    total_gb = sum(f.stat().st_size for f in sf_files) / 1e9
    print(f'Download complete: {len(sf_files)} safetensors file(s), {total_gb:.1f} GB')

print('\nFiles in model dir:')
for f in sorted(MODEL_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)')

## Cell 5 — Launch vLLM server

Starts the OpenAI-compatible API server in the background.
Model loading takes 30–90 s. Cell 6 waits until the server is ready.

In [ ]:
import subprocess, sys, time, os

# Kill any previous vLLM process
subprocess.run('pkill -f "vllm.entrypoints.openai" || true', shell=True)
time.sleep(2)

LOG_FILE = '/tmp/vllm_server.log'

# libcudart.so.13 is bundled under nvidia/cu13 — add it to the library path
_cu13_lib = '/venv/main/lib/python3.12/site-packages/nvidia/cu13/lib'
env = dict(os.environ)
env['LD_LIBRARY_PATH'] = _cu13_lib + ':' + env.get('LD_LIBRARY_PATH', '')
if HF_TOKEN:
    env['HF_TOKEN'] = HF_TOKEN

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model',              str(MODEL_DIR),
    '--served-model-name', MODEL_NAME,
    '--host',              '0.0.0.0',
    '--port',              str(PORT),
    '--max-model-len',     str(MAX_SEQ_LEN),
    '--dtype',             'bfloat16',
    '--trust-remote-code',
    '--gpu-memory-utilization', '0.90',
    '--override-generation-config', '{"enable_thinking": false}',
]

with open(LOG_FILE, 'w') as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

print(f'vLLM server started  PID={proc.pid}')
print(f'Log: {LOG_FILE}')
print(f'LD_LIBRARY_PATH includes: {_cu13_lib}')
print(f'Waiting for model to load (run Cell 6 next) ...')
print(f'\nFull command:')
print(' '.join(cmd))

## Cell 6 — Wait for server ready

In [ ]:
import time, urllib.request, json

TIMEOUT_S = 180
deadline  = time.time() + TIMEOUT_S

print('Waiting for vLLM to load model', end='', flush=True)
while time.time() < deadline:
    try:
        with urllib.request.urlopen(f'http://localhost:{PORT}/health', timeout=5) as r:
            if r.status == 200:
                print(f'  ready!')
                break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(6)
else:
    print('\n\n=== vLLM log (last 60 lines) ===')
    with open(LOG_FILE) as f:
        lines = f.readlines()
    print(''.join(lines[-60:]))
    raise RuntimeError(f'Server did not start within {TIMEOUT_S}s — check log above.')

# List served models
with urllib.request.urlopen(f'http://localhost:{PORT}/v1/models') as r:
    models = json.loads(r.read())
ids = [m['id'] for m in models['data']]
print(f'Served models: {ids}')
print(f'\nApp env vars to set:')
print(f'  OLLAMA_BASE_URL = http://localhost:{PORT}/v1')
print(f'  OLLAMA_MODEL    = {MODEL_NAME}')

## Cell 7 — Basic inference test

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=f'http://localhost:{PORT}/v1',
    api_key='not-required',
)

resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': 'What is your name and what can you help with?'}],
    max_tokens=200,
    temperature=0.0,
)
print('=== Basic response ===')
print(resp.choices[0].message.content)
print(f'\nUsage: {resp.usage}')

## Cell 8 — Tool call test

Gemma 4 uses native `<tool_code>` / `<tool_response>` format in its raw output.
vLLM may return this as raw text in `message.content` (not in `tool_calls`).
The ARIA app's `orchestrator.py` has a fallback raw-text parser that handles both cases.

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'threshold_tuning',
        'description': 'Run FP/FN threshold tuning analysis for a customer segment and attribute.',
        'parameters': {
            'type': 'object',
            'properties': {
                'segment':   {'type': 'string', 'description': 'Customer segment (e.g. Business)'},
                'attribute': {'type': 'string', 'description': 'Transaction attribute (e.g. Avg Weekly Transactions)'},
            },
            'required': ['segment', 'attribute'],
        },
    },
}]

resp = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{
        'role': 'user',
        'content': 'Show FP/FN trade-off for Business customers by average weekly transactions.'
    }],
    tools=tools,
    max_tokens=300,
    temperature=0.0,
)
msg = resp.choices[0].message

print('=== Tool call response ===')
if msg.tool_calls:
    print('Format: OpenAI tool_calls (vLLM parsed successfully)')
    for tc in msg.tool_calls:
        print(f'  {tc.function.name}({tc.function.arguments})')
else:
    print('Format: raw text (Gemma 4 native — app fallback parser will handle this)')
    print(msg.content[:600])

print(f'\nFinish reason: {resp.choices[0].finish_reason}')

## Connect ARIA App to vLLM

### Option A — Direct IP (vast.ai port forward)

In vast.ai, add port `8000` to the exposed ports list for your instance.
Then on your **local Windows machine**:

```powershell
$env:OLLAMA_BASE_URL = "http://<vast-ai-ip>:8000/v1"
$env:OLLAMA_MODEL    = "aria-v16"
.venv\Scripts\python.exe application.py
```

### Option B — Cloudflare tunnel (recommended for demos)

On the **vast.ai instance**:
```bash
cloudflared tunnel --url http://localhost:8000
```
Copy the `https://<random>.trycloudflare.com` URL, then:
```powershell
$env:OLLAMA_BASE_URL = "https://<random>.trycloudflare.com/v1"
$env:OLLAMA_MODEL    = "aria-v16"
.venv\Scripts\python.exe application.py
```

> **Note:** vLLM uses port 8000 (not Ollama's 11434). The `/v1` suffix is required.
> Both Ollama and vLLM expose the same `/v1/chat/completions` OpenAI endpoint —
> `OLLAMA_BASE_URL` is passed directly as the OpenAI client `base_url`.

## Cell 9 — (Optional) Cloudflare tunnel

In [ ]:
import subprocess, shutil, time

if not shutil.which('cloudflared'):
    print('Installing cloudflared ...')
    subprocess.run(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
        'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && '
        'chmod +x /usr/local/bin/cloudflared',
        shell=True, check=True
    )

cf_log = '/tmp/cloudflared.log'
subprocess.run('pkill -f cloudflared || true', shell=True)
time.sleep(1)

with open(cf_log, 'w') as f:
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
        stdout=f, stderr=subprocess.STDOUT
    )

print('Waiting for tunnel URL ...', end='', flush=True)
import re
deadline = time.time() + 30
url = None
while time.time() < deadline:
    time.sleep(2)
    log_text = open(cf_log).read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if m:
        url = m.group(0)
        break
    print('.', end='', flush=True)

if url:
    print(f'\n\nTunnel URL: {url}')
    print(f'Set on local machine:')
    print(f'  $env:OLLAMA_BASE_URL = "{url}/v1"')
    print(f'  $env:OLLAMA_MODEL    = "{MODEL_NAME}"')
else:
    print('\nTimeout — check log:')
    print(open(cf_log).read()[-500:])

## Tail server log (debug)

In [ ]:
with open('/tmp/vllm_server.log') as f:
    lines = f.readlines()
print(''.join(lines[-50:]))